# 07. 동차좌표계와 변환행렬 — SE(3)

$$T = \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix} \in SE(3)$$

05에서 회전행렬($R$)은 다뤘는데, 실제 로봇 팔은 회전만 하는 게 아니라 **이동(translation)도 같이** 일어난다.
그걸 하나의 $4 \times 4$ 행렬로 묶은 게 동차 변환행렬.

- $R \in SO(3)$ : 회전 ($3 \times 3$)
- $t \in \mathbb{R}^3$ : 이동 벡터
- $T_1 T_2$ : 변환의 합성 (좌표계 체인)

**로보틱스 연결:**
- 각 관절마다 $T_i$ 하나씩 → 순기구학 = $T_1 T_2 \cdots T_n$
- ROS2 `geometry_msgs/Transform` 이 이 구조
- DH 파라미터로 $T_i$ 를 체계적으로 만드는 게 표준 방법

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
os.makedirs('assets', exist_ok=True)

plt.rcParams['font.family'] = 'Nanum Gothic'
plt.rcParams['axes.unicode_minus'] = False

def Rx(deg):
    a = np.radians(deg)
    return np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])
def Ry(deg):
    a = np.radians(deg)
    return np.array([[np.cos(a),0,np.sin(a)],[0,1,0],[-np.sin(a),0,np.cos(a)]])
def Rz(deg):
    a = np.radians(deg)
    return np.array([[np.cos(a),-np.sin(a),0],[np.sin(a),np.cos(a),0],[0,0,1]])

def make_T(R, t):
    T = np.eye(4)
    T[:3,:3] = R
    T[:3, 3] = t
    return T

def T_inv(T):
    # T^{-1} = [R^T  -R^T t; 0 1]  — np.inv 보다 수치 안정
    R = T[:3,:3]; t = T[:3,3]
    T_i = np.eye(4)
    T_i[:3,:3] = R.T
    T_i[:3, 3] = -R.T @ t
    return T_i

def draw_frame(ax, T, scale=0.3, label='', alpha=1.0):
    origin = T[:3,3]
    colors = ['#E85D24','#1D9E75','#534AB7']
    axes_label = ['x','y','z']
    for i, (c, l) in enumerate(zip(colors, axes_label)):
        vec = T[:3,i] * scale
        ax.quiver(*origin, *vec, color=c, lw=2, arrow_length_ratio=0.25, alpha=alpha)
    if label:
        ax.text(*(origin + 0.05), label, fontsize=9, fontweight='bold')

## 1. SE(3) 기본 연산 — 합성과 역변환

변환 합성: $T_{A \to C} = T_{A \to B} \cdot T_{B \to C}$

역변환: $T^{-1} = \begin{bmatrix} R^T & -R^T t \\ 0 & 1 \end{bmatrix}$

$R^T = R^{-1}$ 이니까 역행렬 대신 전치만 써도 됨 — 계산이 싸다.

In [ ]:
# 좌표계 체인 예시
T_base_to_1 = make_T(Rz(45), np.array([1., 0., 0.]))
T_1_to_2    = make_T(Ry(30), np.array([0., 0., 1.]))
T_base_to_2 = T_base_to_1 @ T_1_to_2

# 역변환 검증
T_inv_check = T_inv(T_base_to_1)
print('T @ T_inv (단위행렬 확인):')
print((T_base_to_1 @ T_inv_check).round(6))

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

draw_frame(ax, np.eye(4), scale=0.4, label='Base', alpha=0.3)
draw_frame(ax, T_base_to_1, scale=0.4, label='Frame 1')
draw_frame(ax, T_base_to_2, scale=0.4, label='Frame 2')

# 각 원점 연결
origins = [np.array([0,0,0]), T_base_to_1[:3,3], T_base_to_2[:3,3]]
xs, ys, zs = zip(*origins)
ax.plot(xs, ys, zs, 'k--', lw=1.2, alpha=0.4)

ax.set_xlim(-0.5, 2); ax.set_ylim(-1, 1.5); ax.set_zlim(-0.5, 1.5)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('SE(3) 좌표계 체인\nBase → Frame1 → Frame2', fontsize=12)

# 범례용 패치
from matplotlib.lines import Line2D
legend = [Line2D([0],[0],color=c,lw=2,label=l)
          for c,l in [('#E85D24','x축'),('#1D9E75','y축'),('#534AB7','z축')]]
ax.legend(handles=legend, fontsize=9)
plt.tight_layout()
plt.savefig('assets/07_se3_frames.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. DH 파라미터 — 관절 변환을 체계적으로 만드는 방법

관절 $i$ 의 변환행렬을 4개 파라미터로 정의:

| 파라미터 | 의미 |
|----------|------|
| $a_i$ | 링크 길이 (x축 방향) |
| $\alpha_i$ | 링크 비틀림 (x축 회전) |
| $d_i$ | 링크 오프셋 (z축 이동) |
| $\theta_i$ | 관절 각도 (z축 회전) ← **실제 움직이는 값** |

$$T_i = R_z(\theta_i) \cdot T_z(d_i) \cdot T_x(a_i) \cdot R_x(\alpha_i)$$

2R 평면 팔은 $d=0$, $\alpha=0$ 이라 식이 단순해진다.

In [ ]:
def dh_transform(theta_deg, d, a, alpha_deg):
    theta = np.radians(theta_deg)
    alpha = np.radians(alpha_deg)
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [ 0,     sa,     ca,    d],
        [ 0,      0,      0,    1]
    ])

# 2-link 팔 DH 테이블
# joint | theta  | d | a  | alpha
# 1     | theta1 | 0 | L1 | 0
# 2     | theta2 | 0 | L2 | 0
L1, L2 = 1.0, 0.8

def fk_2link(theta1_deg, theta2_deg):
    T1 = dh_transform(theta1_deg,    0, L1, 0)
    T2 = dh_transform(theta2_deg,    0, L2, 0)
    T_ee = T1 @ T2
    return T1, T_ee

# 여러 자세 시각화
configs = [
    (0,   0,   '기본 자세'),
    (45,  45,  'θ1=45°, θ2=45°'),
    (90, -90,  'θ1=90°, θ2=-90°'),
    (30,  60,  'θ1=30°, θ2=60°'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, (t1, t2, title) in zip(axes.flat, configs):
    T1, T_ee = fk_2link(t1, t2)

    origin = np.array([0, 0])
    j1     = T1[:2, 3]
    ee     = T_ee[:2, 3]

    # 링크
    ax.plot([origin[0], j1[0]], [origin[1], j1[1]],
            'o-', color='#534AB7', lw=3, markersize=10, label='Link 1')
    ax.plot([j1[0], ee[0]], [j1[1], ee[1]],
            'o-', color='#1D9E75', lw=3, markersize=10, label='Link 2')
    ax.plot(*ee, '*', color='#E85D24', markersize=15, label=f'EE ({ee[0]:.2f}, {ee[1]:.2f})')

    # 작업 공간 원 (참고용)
    circle_outer = plt.Circle((0,0), L1+L2, fill=False, color='gray', lw=0.8, linestyle='--', alpha=0.4)
    circle_inner = plt.Circle((0,0), abs(L1-L2), fill=False, color='gray', lw=0.8, linestyle='--', alpha=0.4)
    ax.add_patch(circle_outer); ax.add_patch(circle_inner)

    ax.set_xlim(-2.2, 2.2); ax.set_ylim(-2.2, 2.2)
    ax.set_aspect('equal'); ax.grid(True, alpha=0.25)
    ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f'{title}\n엔드이펙터: ({ee[0]:.3f}, {ee[1]:.3f})', fontsize=11)
    ax.legend(fontsize=9, loc='upper right')

plt.suptitle('2R 팔 순기구학 — DH 파라미터로 구현', fontsize=14, y=1.01, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/07_fk_2link.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 작업 공간 (Workspace) 시각화

모든 $\theta_1, \theta_2 \in [-180°, 180°]$ 에 대해 엔드이펙터가 도달할 수 있는 공간.

In [ ]:
thetas = np.linspace(-np.pi, np.pi, 150)
t1_grid, t2_grid = np.meshgrid(thetas, thetas)

x_ee = L1*np.cos(t1_grid) + L2*np.cos(t1_grid + t2_grid)
y_ee = L1*np.sin(t1_grid) + L2*np.sin(t1_grid + t2_grid)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))

ax = axes[0]
ax.scatter(x_ee.ravel(), y_ee.ravel(), s=0.3, alpha=0.15, color='#534AB7')
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title(f'2R 팔 작업 공간\nL1={L1}, L2={L2}', fontsize=12)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')

circle_outer = plt.Circle((0,0), L1+L2, fill=False, color='#E85D24', lw=2, label=f'최대 반경 {L1+L2}m')
circle_inner = plt.Circle((0,0), abs(L1-L2), fill=False, color='#1D9E75', lw=2, label=f'최소 반경 {abs(L1-L2)}m')
ax.add_patch(circle_outer); ax.add_patch(circle_inner)
ax.legend(fontsize=10)

# T 행렬 구조 시각화
ax2 = axes[1]
T_example = fk_2link(45, 30)[1]
im = ax2.imshow(np.abs(T_example), cmap='Blues', vmin=0, vmax=1.5)
for i in range(4):
    for j in range(4):
        val = T_example[i,j]
        ax2.text(j, i, f'{val:.3f}', ha='center', va='center',
                 fontsize=11, color='white' if abs(val) > 0.7 else 'black')
ax2.set_title('T_base→EE 행렬 (θ1=45°, θ2=30°)\n왼쪽 3×3 = 회전 / 4열 = 위치', fontsize=11)
ax2.set_xticks(range(4)); ax2.set_yticks(range(4))
ax2.set_xticklabels(['col0','col1','col2','t']); ax2.set_yticklabels(['row0','row1','row2','[0,1]'])
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.savefig('assets/07_workspace.png', dpi=150, bbox_inches='tight')
plt.show()

print('T_base→EE:')
print(T_example.round(4))
print(f'\n위치: ({T_example[0,3]:.4f}, {T_example[1,3]:.4f})')
print(f'det(R): {np.linalg.det(T_example[:3,:3]):.6f}  (1이어야 함)')

## 요약

| 개념 | 수식 | 로보틱스 활용 |
|------|------|---------------|
| SE(3) | $T = [R\ t;\ 0\ 1]$ | 관절 좌표계 표현 |
| 변환 합성 | $T_1 T_2$ | 순기구학 체인 |
| 역변환 | $T^{-1} = [R^T\ {-R^T t};\ 0\ 1]$ | 역기구학, 좌표 역변환 |
| DH 파라미터 | $T_i(	heta_i, d_i, a_i, \alpha_i)$ | 표준 기구학 기술 방법 |

**다음 노트북:** `08_jacobian.ipynb` — 야코비안과 미분기구학